# 08. Temporal Cross-Validation and Leakage Sensitivity

This notebook isolates the professor's cross-validation concern:

> Does the leave-one-tournament-out cross-validation design train on tournaments that occur after the tournament being predicted?

The answer is yes for most LOTO folds. This notebook explores two leakage-safe alternatives using the same locked feature sets and model families from `06_final_report_model_lock.ipynb`.

**Option A: Expanding-window forward validation**

- For each test tournament, train only on earlier tournaments.
- This is a leakage-safe cross-validation sensitivity check.

**Option B: 2018 validation + 2022 final test**

- Train through 2014 and evaluate on 2018 as a validation tournament.
- Then freeze the design, refit through 2018, and evaluate on 2022 as a final prospective test.
- Also report a supplemental no-refit 2022 check trained only through 2014.

This is an exploratory audit. The goal is to understand whether the core report conclusions survive stricter temporal validation.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
import matplotlib
if "get_ipython" not in globals():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)

NOTEBOOK_DIR = Path.cwd()
MODEL_LOCK_NB = NOTEBOOK_DIR / "06_final_report_model_lock.ipynb"
OUT_DIR = NOTEBOOK_DIR / "data" / "cv_leakage_audit"
FIG_DIR = NOTEBOOK_DIR / "docs" / "figures" / "cv_leakage_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_KEYS = ["baseline", "team_history", "player_enriched"]
MODEL_NAMES = {
    "baseline": "Baseline",
    "team_history": "Team-history",
    "player_enriched": "Player-enriched",
}
CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = {0: "fav_loses", 1: "draw", 2: "fav_wins"}

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("MODEL_LOCK_NB exists:", MODEL_LOCK_NB.exists())

## 1. Reuse the Locked Modeling Context

To keep this audit comparable to the report, we execute only the setup/configuration cells from `06_final_report_model_lock.ipynb`:

- imports and paths;
- rebuilt favorite/underdog modeling table;
- locked feature sets;
- locked model configurations;
- helper metric functions.

No final report metrics are imported as conclusions here; they are recomputed under stricter temporal splits below.

In [ ]:
def exec_lock_cells(cell_indices):
    nb = json.loads(MODEL_LOCK_NB.read_text())
    for idx in cell_indices:
        cell = nb["cells"][idx]
        assert cell["cell_type"] == "code", f"Cell {idx} is not code"
        print(f"Executing 06 cell {idx}...")
        exec(compile("".join(cell["source"]), f"{MODEL_LOCK_NB.name}:cell{idx}", "exec"), globals())


# 1 imports/paths, 3 modeling table, 5 feature sets, 7 final configs, 9 helper functions.
exec_lock_cells([1, 3, 5, 7, 9])

# Cell 1 from the lock notebook defines CLASS_LABELS as strings for display.
# This audit uses numeric labels for sklearn metrics.
CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = {0: "fav_loses", 1: "draw", 2: "fav_wins"}

all_years = sorted(df_fav["year"].dropna().astype(int).unique())
print("Available years:", all_years)
print("df_fav shape:", df_fav.shape)
display(df_fav.groupby("year")["y"].value_counts().unstack(fill_value=0).rename(columns=CLASS_NAMES))

## 2. Metric Helpers

The helpers below use the locked model pipelines but control the train/test years explicitly.

In [ ]:
def label_counts(y):
    counts = pd.Series(y).value_counts().reindex(CLASS_LABELS, fill_value=0)
    return {f"n_{CLASS_NAMES[k]}": int(v) for k, v in counts.items()}


def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion": confusion_matrix(y_true, y_pred, labels=CLASS_LABELS).tolist(),
        **label_counts(y_true),
    }


def fit_predict_years(model_key, train_years, test_years):
    cfg = FINAL_CONFIGS[model_key]
    train_years = sorted(int(y) for y in train_years)
    test_years = sorted(int(y) for y in test_years)

    train = df_fav[df_fav["year"].isin(train_years)].copy()
    test = df_fav[df_fav["year"].isin(test_years)].copy()
    if train.empty:
        raise ValueError(f"No training rows for {model_key}: {train_years}")
    if test.empty:
        raise ValueError(f"No test rows for {model_key}: {test_years}")

    model = clone(cfg["pipeline"])
    model.fit(train[cfg["features"]], train["y"])
    pred = model.predict(test[cfg["features"]])
    prob = model.predict_proba(test[cfg["features"]])

    pred_df = test[[
        "match_id", "tournament_id", "year", "y", "group_stage", "knockout_stage",
        "fav_team_id", "und_team_id",
    ]].copy()
    pred_df["model_key"] = model_key
    pred_df["pred"] = pred
    pred_df["prob_fav_loses"] = prob[:, 0]
    pred_df["prob_draw"] = prob[:, 1]
    pred_df["prob_fav_wins"] = prob[:, 2]
    pred_df["correct"] = pred_df["pred"] == pred_df["y"]

    metrics = evaluate_predictions(pred_df["y"], pred_df["pred"])
    metrics.update({
        "model_key": model_key,
        "model_name": MODEL_NAMES[model_key],
        "model_family": cfg["model_family"],
        "train_min_year": cfg["train_min_year"],
        "train_years": ",".join(map(str, train_years)),
        "test_years": ",".join(map(str, test_years)),
        "n_train": int(len(train)),
        "n_test": int(len(test)),
    })
    return metrics, pred_df


def summarize_metric_table(rows, group_cols):
    df = pd.DataFrame(rows)
    summary = (
        df.groupby(group_cols)
        .agg(
            n_folds=("accuracy", "size"),
            total_test_matches=("n_test", "sum"),
            mean_accuracy=("accuracy", "mean"),
            sd_accuracy=("accuracy", "std"),
            weighted_accuracy=("accuracy", lambda s: np.average(s, weights=df.loc[s.index, "n_test"])),
            mean_macro_f1=("macro_f1", "mean"),
            sd_macro_f1=("macro_f1", "std"),
            weighted_macro_f1=("macro_f1", lambda s: np.average(s, weights=df.loc[s.index, "n_test"])),
        )
        .reset_index()
    )
    return summary

## 3. Leakage Audit of Current LOTO Design

This table documents the issue: LOTO cross-validation uses all other pre-holdout tournaments as training data. When the held-out fold is an earlier tournament, later tournaments enter the training set.

In [ ]:
leakage_rows = []
for model_key, cfg in FINAL_CONFIGS.items():
    pre_holdout_years = [
        year for year in all_years
        if cfg["train_min_year"] <= year and year not in HOLDOUT_YEARS
    ]
    for heldout_year in pre_holdout_years:
        loto_train_years = [year for year in pre_holdout_years if year != heldout_year]
        future_years = [year for year in loto_train_years if year > heldout_year]
        past_years = [year for year in loto_train_years if year < heldout_year]
        leakage_rows.append({
            "model_key": model_key,
            "model_name": MODEL_NAMES[model_key],
            "heldout_year": heldout_year,
            "loto_train_years": ",".join(map(str, loto_train_years)),
            "uses_future_tournaments": bool(future_years),
            "future_train_years": ",".join(map(str, future_years)),
            "past_train_years": ",".join(map(str, past_years)),
            "forward_chaining_possible": bool(past_years),
        })

leakage_audit = pd.DataFrame(leakage_rows)
leakage_audit.to_csv(OUT_DIR / "current_loto_leakage_audit.csv", index=False)
display(leakage_audit)
display(
    leakage_audit.groupby("model_name")["uses_future_tournaments"]
    .agg(folds_using_future_tournaments="sum", total_loto_folds="count")
    .reset_index()
)

## 4. Option A: Expanding-Window Forward Validation

For each tournament, train only on earlier tournaments in that model's allowed training window.

This avoids temporal leakage completely, but it also exposes the small-data problem: the first eligible fold has very little training data, especially for the 2006+ baseline and player-enriched models.

In [ ]:
forward_rows = []
forward_pred_frames = []

for model_key, cfg in FINAL_CONFIGS.items():
    eligible_years = [
        year for year in all_years
        if cfg["train_min_year"] <= year and year not in HOLDOUT_YEARS
    ]
    for test_year in eligible_years:
        train_years = [year for year in eligible_years if year < test_year]
        if not train_years:
            continue
        metrics, pred_df = fit_predict_years(model_key, train_years, [test_year])
        metrics.update({
            "validation_option": "A_forward_expanding_window",
            "test_year": test_year,
        })
        forward_rows.append(metrics)
        pred_df["validation_option"] = "A_forward_expanding_window"
        pred_df["train_years"] = metrics["train_years"]
        forward_pred_frames.append(pred_df)

forward_folds = pd.DataFrame(forward_rows)
forward_preds = pd.concat(forward_pred_frames, ignore_index=True)
forward_summary = summarize_metric_table(
    forward_rows,
    ["validation_option", "model_key", "model_name", "model_family", "train_min_year"],
)

forward_folds.to_csv(OUT_DIR / "option_a_forward_validation_folds.csv", index=False)
forward_summary.to_csv(OUT_DIR / "option_a_forward_validation_summary.csv", index=False)
forward_preds.to_csv(OUT_DIR / "option_a_forward_validation_predictions.csv", index=False)

forward_stage_rows = []
forward_preds["stage"] = np.where(forward_preds["knockout_stage"].astype(bool), "knockout_stage", "group_stage")
for keys, sub in forward_preds.groupby(["model_key", "validation_option", "train_years", "year", "stage"]):
    model_key, validation_option, train_years, test_year, stage = keys
    metrics = evaluate_predictions(sub["y"], sub["pred"])
    metrics.update({
        "model_key": model_key,
        "model_name": MODEL_NAMES[model_key],
        "validation_option": validation_option,
        "train_years": train_years,
        "test_year": int(test_year),
        "stage": stage,
        "n_test": int(len(sub)),
    })
    forward_stage_rows.append(metrics)
forward_stage_folds = pd.DataFrame(forward_stage_rows)
forward_stage_folds.to_csv(OUT_DIR / "option_a_forward_validation_stage_folds.csv", index=False)

display(forward_folds[[
    "model_name", "test_year", "train_years", "n_train", "n_test",
    "accuracy", "macro_f1", "n_fav_loses", "n_draw", "n_fav_wins",
]].sort_values(["model_name", "test_year"]))
display(forward_summary.round(3))

### Option A Figure

Fold-level metrics are shown because the number of folds is small. The point is not to claim a precise CV estimate; it is to check whether the old LOTO story survives a leakage-safe direction-of-time test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
colors = {"baseline": "#6B7280", "team_history": "#2563EB", "player_enriched": "#DC2626"}

for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
    for model_key in MODEL_KEYS:
        sub = forward_folds[forward_folds["model_key"] == model_key].sort_values("test_year")
        ax.plot(sub["test_year"], sub[metric], marker="o", color=colors[model_key], label=MODEL_NAMES[model_key])
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylim(0, 1)
    ax.set_xlabel("Test tournament")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Metric value")
axes[1].legend(frameon=False, loc="lower left")
fig.suptitle("Option A: Expanding-window forward validation")
fig.tight_layout()
fig.savefig(FIG_DIR / "option_a_forward_validation_metrics.png", dpi=200)
plt.show()

## 5. Option B: 2018 Validation + 2022 Final Test

This is a cleaner report-style evaluation design:

1. Train using tournaments available through 2014.
2. Evaluate 2018 as a validation tournament.
3. Freeze the design.
4. Refit using tournaments through 2018.
5. Evaluate 2022 as final prospective test.

The notebook also reports a supplemental no-refit 2022 check trained only through 2014, so we can separate the effect of adding 2018 training data from the effect of the 2022 tournament itself.

In [ ]:
option_b_rows = []
option_b_pred_frames = []

for model_key, cfg in FINAL_CONFIGS.items():
    pre_2018_train_years = [
        year for year in all_years
        if cfg["train_min_year"] <= year <= 2014
    ]
    through_2018_train_years = [
        year for year in all_years
        if cfg["train_min_year"] <= year <= 2018
    ]

    experiments = [
        ("B1_validate_2018_train_through_2014", pre_2018_train_years, [2018], "validation"),
        ("B2_final_2022_refit_through_2018", through_2018_train_years, [2022], "final_test"),
        ("B3_supplement_2022_no_refit_train_through_2014", pre_2018_train_years, [2022], "supplemental_no_refit"),
    ]

    for experiment, train_years, test_years, split_role in experiments:
        metrics, pred_df = fit_predict_years(model_key, train_years, test_years)
        metrics.update({
            "validation_option": "B_2018_validation_2022_final",
            "experiment": experiment,
            "split_role": split_role,
            "test_year": test_years[0],
        })
        option_b_rows.append(metrics)
        pred_df["validation_option"] = "B_2018_validation_2022_final"
        pred_df["experiment"] = experiment
        pred_df["split_role"] = split_role
        pred_df["train_years"] = metrics["train_years"]
        option_b_pred_frames.append(pred_df)

option_b_results = pd.DataFrame(option_b_rows)
option_b_preds = pd.concat(option_b_pred_frames, ignore_index=True)

option_b_results.to_csv(OUT_DIR / "option_b_2018_validation_2022_final_results.csv", index=False)
option_b_preds.to_csv(OUT_DIR / "option_b_2018_validation_2022_final_predictions.csv", index=False)

option_b_stage_rows = []
option_b_preds["stage"] = np.where(option_b_preds["knockout_stage"].astype(bool), "knockout_stage", "group_stage")
for keys, sub in option_b_preds.groupby(["experiment", "split_role", "model_key", "train_years", "stage"]):
    experiment, split_role, model_key, train_years, stage = keys
    metrics = evaluate_predictions(sub["y"], sub["pred"])
    metrics.update({
        "experiment": experiment,
        "split_role": split_role,
        "model_key": model_key,
        "model_name": MODEL_NAMES[model_key],
        "train_years": train_years,
        "stage": stage,
        "n_test": int(len(sub)),
    })
    option_b_stage_rows.append(metrics)
option_b_stage_results = pd.DataFrame(option_b_stage_rows)
option_b_stage_results.to_csv(OUT_DIR / "option_b_stage_results.csv", index=False)

display(option_b_results[[
    "experiment", "split_role", "model_name", "train_years", "test_years",
    "n_train", "n_test", "accuracy", "macro_f1", "n_fav_loses", "n_draw", "n_fav_wins",
]].sort_values(["experiment", "model_name"]))
display(option_b_stage_results[[
    "experiment", "split_role", "stage", "model_name", "n_test", "accuracy", "macro_f1",
]].sort_values(["experiment", "stage", "model_name"]))

## 6. Option B Model Differences

The most report-relevant comparison is player-enriched minus team-history. Positive values favor the player-enriched model.

In [ ]:
diff_rows = []
for experiment, sub in option_b_results.groupby("experiment"):
    wide = sub.set_index("model_key")
    for metric in ["accuracy", "macro_f1"]:
        diff_rows.append({
            "experiment": experiment,
            "metric": metric,
            "player_minus_team_history": float(wide.loc["player_enriched", metric] - wide.loc["team_history", metric]),
            "player_value": float(wide.loc["player_enriched", metric]),
            "team_history_value": float(wide.loc["team_history", metric]),
            "baseline_value": float(wide.loc["baseline", metric]),
        })

option_b_differences = pd.DataFrame(diff_rows)
option_b_differences.to_csv(OUT_DIR / "option_b_player_minus_team_history_differences.csv", index=False)
display(option_b_differences.round(3))

In [ ]:
plot_df = option_b_results.copy()
experiment_order = [
    "B1_validate_2018_train_through_2014",
    "B2_final_2022_refit_through_2018",
    "B3_supplement_2022_no_refit_train_through_2014",
]
experiment_labels = {
    "B1_validate_2018_train_through_2014": "2018 validation\ntrain<=2014",
    "B2_final_2022_refit_through_2018": "2022 final\ntrain<=2018",
    "B3_supplement_2022_no_refit_train_through_2014": "2022 no-refit\ntrain<=2014",
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
width = 0.22
x_base = np.arange(len(experiment_order))
offsets = {"baseline": -width, "team_history": 0, "player_enriched": width}

for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
    for model_key in MODEL_KEYS:
        sub = plot_df[plot_df["model_key"] == model_key].set_index("experiment").reindex(experiment_order)
        ax.bar(x_base + offsets[model_key], sub[metric], width=width, color=colors[model_key], label=MODEL_NAMES[model_key])
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)
    ax.set_xticks(x_base)
    ax.set_xticklabels([experiment_labels[e] for e in experiment_order])
axes[0].set_ylabel("Metric value")
axes[1].legend(frameon=False, loc="upper right")
fig.suptitle("Option B: 2018 validation and 2022 final test")
fig.tight_layout()
fig.savefig(FIG_DIR / "option_b_2018_2022_metrics.png", dpi=200)
plt.show()

## 7. Compare Against Locked Report Holdout

The current report trains through 2014 and pools 2018+2022 as the final holdout. This section prints the locked pooled result next to the stricter alternatives for interpretation.

In [ ]:
locked_table1 = pd.read_csv(LOCK_DIR / "table1_summary.csv")
locked_display = locked_table1[[
    "model_key", "display_name", "cv_accuracy", "holdout_accuracy", "holdout_macro_f1"
]].copy()
locked_display["source"] = "locked_report_loto_cv_and_2018_2022_pooled_holdout"
locked_display.to_csv(OUT_DIR / "locked_report_reference_metrics.csv", index=False)
display(locked_display.round(3))

## 8. Initial Interpretation Checklist

Use this checklist after reviewing the tables:

1. Does Option A still show player-enriched as stronger than team-history?
2. Does Option B's 2022 final test support or weaken the knockout/player-data claim?
3. Are any differences large enough to justify strong wording, or should the report frame them as suggestive?
4. Is the safest report design to replace LOTO CV entirely, or to move LOTO into an appendix and use Option B as the main validation story?

Recommended reporting language if leakage-safe validation is adopted:

> The original leave-one-tournament-out check is useful as a stability diagnostic, but it is not a prospective validation because some folds train on later tournaments. We therefore added leakage-safe temporal checks: expanding-window validation over past tournaments and a 2018-validation / 2022-final-test split. These checks are noisier because World Cups are small, but they better match the forecasting use case.